In [1]:
import pandas as pd
import numpy as np
import pyarrow
from src import *

from sklearn.model_selection import  train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, MinMaxScaler

import keras
from keras.models import Sequential
from keras.layers import Dense, Input, Dropout
from keras import optimizers

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras import regularizers
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.optimizers import Adam

# Configurações 
print(tf.config.list_physical_devices())
BATCH_SIZE = 1024

[PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]


In [1]:
def num_max_neuronio(X, d):
    CT = len(X)
    return int((CT - 10)/(10 * (d + 2)))

# 1.0 - Classificação de Presença Baseada em Fatores Socioeconômicos Usando Redes Neurais

In [2]:
colunas = ['Q001','Q002','Q003', 'Q004', 'Q005', 'Q006', 'Q007', 'Q008', 'Q009', 'Q010', 'Q011', 'Q012', 'Q013', 'Q014', 'Q015', 'Q016', 'Q017', 'Q018', 
           'Q019', 'Q020', 'Q021', 'Q022', 'Q023', 'Q024', 'Q025', 'TP_PRESENCA_LC', 'TP_PRESENCA_CH', 'TP_PRESENCA_CN', 'TP_PRESENCA_MT', 
           'TP_FAIXA_ETARIA', 'TP_SEXO','TP_ESTADO_CIVIL', 'TP_COR_RACA', 'TP_ESCOLA', 'TP_ST_CONCLUSAO', 'IN_TREINEIRO', 
           'NU_ANO', 'TP_LOCALIZACAO_ESC','TP_SIT_FUNC_ESC', 'NU_NOTA_CN', 'NU_NOTA_CH', 'NU_NOTA_LC', 'NU_NOTA_MT', 'NU_NOTA_REDACAO', 'TP_DEPENDENCIA_ADM_ESC']

df = pd.read_parquet("enem_parquet", columns = colunas)

## 1.1 - Criando uma Amostra

In [3]:
df = df.sample(n=1_000_000, random_state=42)

## 1.2- Limpando os Dados

In [4]:
FALTOU = (
    (df['TP_PRESENCA_CH'] != 1) | 
    (df['TP_PRESENCA_LC'] != 1) | 
    (df['TP_PRESENCA_CN'] != 1) | 
    (df['TP_PRESENCA_MT'] != 1)
)

df['FALTOU'] = FALTOU.astype(int)

In [5]:
df = df[df['FALTOU'] == 0]

In [6]:
df = df[df['TP_ESCOLA'].isin([2,3])]
df = df[df['TP_ESTADO_CIVIL'].isin([1,2,3,4])]

In [7]:
df['TP_LOCALIZACAO_ESC'] = df['TP_LOCALIZACAO_ESC'].fillna(0)
df['TP_DEPENDENCIA_ADM_ESC'] = df['TP_DEPENDENCIA_ADM_ESC'].fillna(0)
df['TP_SIT_FUNC_ESC'] = df['TP_SIT_FUNC_ESC'].fillna(0)

In [23]:
questionario = [f'Q{str(i).zfill(3)}' for i in range(1, 26) if i != 5]
df = df.dropna(subset=questionario)

## 1.3 - Criando Rótulo

In [22]:
df['MEDIA'] = (df['NU_NOTA_CN'] + df['NU_NOTA_CH'] + df['NU_NOTA_MT']+  df['NU_NOTA_LC'] + df['NU_NOTA_REDACAO']) / 5

df['CLASSE'] = df.groupby('NU_ANO')['MEDIA'].transform(
    lambda x: pd.qcut(x, q=2, labels=[0,1])
).astype(int)
    
df['CLASSE'] = df['CLASSE'].astype(int)

## 1.4 - Tratando os Dados

In [24]:
nominais = ['TP_ESCOLA', 'TP_DEPENDENCIA_ADM_ESC',
            'TP_ESTADO_CIVIL', 'TP_LOCALIZACAO_ESC', 'TP_SIT_FUNC_ESC']

ordinais = ['TP_FAIXA_ETARIA', 'TP_ST_CONCLUSAO']

categorias_quest = []
for i in range(1, 26):
    if i == 5:
        continue
    elif i == 6:
        categorias_quest.append(list('ABCDEFGHIJKLMNOPQ'))  
    elif i == 25:
        categorias_quest.append(list('AB'))                  
    elif i in [1, 2]:
        categorias_quest.append(list('ABCDEFGH'))            
    elif i in [3, 4]:
        categorias_quest.append(list('ABCDEF'))              
    else:
        categorias_quest.append(list('ABCDE'))               

binarias = ['IN_TREINEIRO']

In [25]:
pipe_nominal = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

pipe_ordinal = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)

pipe_quest = OrdinalEncoder(
    categories=categorias_quest,
    handle_unknown='use_encoded_value',
    unknown_value=-1
)

pipe_continua = MinMaxScaler()

preprocessor = ColumnTransformer(transformers=[
    ('nominal',      pipe_nominal,  nominais),
    ('ordinal',      pipe_ordinal,  ordinais),
    ('questionario', pipe_quest,    questionario),
    ('binaria',      'passthrough', binarias),
], remainder='drop')


## 1.5- Construção da Matriz X e Vetor y

In [26]:
X = preprocessor.fit_transform(df[colunas])
X = X.astype(np.float32)
y = df['CLASSE']

## 1.6 - Separação em Dados de Treino e Teste

In [27]:
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
x_train, x_val, y_train, y_val = train_test_split(x_train, y_train, test_size=0.2, random_state=42)

In [29]:
max_neurons = num_max_neuronio(x_train, d = 1)
print("Número máximo de neurônios:", max_neurons)

Número máximo de neurônios: 4660


## 1.7 - Treinando a Rede Neural

In [36]:
model = Sequential()

model.add(Dense(256, input_dim=x_train.shape[1], kernel_initializer='he_normal', activation='relu'))
model.add(BatchNormalization())
model.add(Dropout(0.2))

model.add(Dense(128, input_dim=x_train.shape[1], kernel_initializer='he_normal', activation='relu'))
model.add(BatchNormalization())
model.add(Dropout(0.2))

model.add(Dense(64, kernel_initializer='he_normal', activation='relu'))
model.add(BatchNormalization())
model.add(Dropout(0.2))

model.add(Dense(1, kernel_initializer='he_normal', activation='sigmoid'))

model.compile(loss='binary_crossentropy', optimizer=Adam(), metrics=['accuracy'])


es = EarlyStopping(monitor='val_loss', mode='min', patience=10, verbose=1)

history = model.fit(
    x_train, y_train,
    epochs=100,
    batch_size= BATCH_SIZE,
    validation_data= (x_val, y_val),
    callbacks=[es],
    verbose=1
)

Epoch 1/100
137/137 ━━━━━━━━━━━━━━━━━━━━ 5s 16ms/step - accuracy: 0.6788 - loss: 0.6116 - val_accuracy: 0.7062 - val_loss: 0.5668
Epoch 2/100
137/137 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - accuracy: 0.7010 - loss: 0.5737 - val_accuracy: 0.7119 - val_loss: 0.5583
Epoch 3/100
137/137 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - accuracy: 0.7070 - loss: 0.5661 - val_accuracy: 0.7112 - val_loss: 0.5570
Epoch 4/100
137/137 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - accuracy: 0.7086 - loss: 0.5630 - val_accuracy: 0.7122 - val_loss: 0.5557
Epoch 5/100
137/137 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - accuracy: 0.7100 - loss: 0.5606 - val_accuracy: 0.7136 - val_loss: 0.5560
Epoch 6/100
137/137 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - accuracy: 0.7112 - loss: 0.5589 - val_accuracy: 0.7128 - val_loss: 0.5548
Epoch 7/100
137/137 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - accuracy: 0.7123 - loss: 0.5581 - val_accuracy: 0.7130 - val_loss: 0.5551
Epoch 8/100
137/137 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - accuracy: 0.7129 - loss: 0.5564 - 

## 1.8 - Exibindo Resultados no Teste

In [37]:
y_pred = (model.predict(x_test) > 0.5).astype(int).flatten()
print(classification_report(y_test, y_pred))

1366/1366 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step
              precision    recall  f1-score   support

           0       0.70      0.76      0.73     21901
           1       0.74      0.66      0.70     21798

    accuracy                           0.71     43699
   macro avg       0.72      0.71      0.71     43699
weighted avg       0.72      0.71      0.71     43699



## Seção de Testes

In [48]:
import numpy as np
import pandas as pd

#np.random.seed(42)  # para reprodutibilidade

x_aluno = pd.DataFrame([{
    # Questionário socioeconômico
    'Q001': np.random.choice(list('ABCDEFGH')),
    'Q002': np.random.choice(list('ABCDEFGH')),
    'Q003': np.random.choice(list('ABCDEF')),
    'Q004': np.random.choice(list('ABCDEF')),
    'Q006': np.random.choice(list('ABCDEFGHIJKLMNOPQ')),
    'Q007': np.random.choice(list('ABCD')),
    'Q008': np.random.choice(list('ABCDE')),
    'Q009': np.random.choice(list('ABCDE')),
    'Q010': np.random.choice(list('ABCDE')),
    'Q011': np.random.choice(list('ABCDE')),
    'Q012': np.random.choice(list('ABCDE')),
    'Q013': np.random.choice(list('ABCDE')),
    'Q014': np.random.choice(list('ABCDE')),
    'Q015': np.random.choice(list('ABCDE')),
    'Q016': np.random.choice(list('ABCDE')),
    'Q017': np.random.choice(list('ABCDE')),
    'Q018': np.random.choice(list('ABCDE')),
    'Q019': np.random.choice(list('ABCDE')),
    'Q020': np.random.choice(list('ABCDE')),
    'Q021': np.random.choice(list('ABCDE')),
    'Q022': np.random.choice(list('ABCDE')),
    'Q023': np.random.choice(list('ABCDE')),
    'Q024': np.random.choice(list('ABCDE')),
    'Q025': np.random.choice(list('AB')),

    # Categóricas
    'TP_FAIXA_ETARIA':        np.random.choice([3,4,2,1,14,6,5,12,8,7,11,9,13,16,10,17,15,19,18,20]),
    'TP_ST_CONCLUSAO':        2,
    'TP_ESCOLA':              np.random.choice([2, 3]),
    'TP_DEPENDENCIA_ADM_ESC': np.random.choice([0., 4., 2., 1., 3.]),
    'TP_ESTADO_CIVIL':        np.random.choice([1, 2, 3, 4]),
    'TP_LOCALIZACAO_ESC':     np.random.choice([0., 1., 2.]),
    'TP_SIT_FUNC_ESC':        np.random.choice([0., 1., 3., 4., 2.]),

    # Binária
    'IN_TREINEIRO': np.random.choice([0, 1]),
}])

# Transformar com o preprocessor já treinado
x_aluno_transformado = preprocessor.transform(x_aluno).astype(np.float32)

# Probabilidade
prob = model.predict(x_aluno_transformado).flatten()[0]
print(f"Probabilidade de desempenho alto: {prob:.1%}")

# Nota estimada
media_classe_0 = df[df['CLASSE'] == 0]['MEDIA'].mean()
media_classe_1 = df[df['CLASSE'] == 1]['MEDIA'].mean()
nota_estimada = (1 - prob) * media_classe_0 + prob * media_classe_1
print(f"Nota estimada: {nota_estimada:.1f}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
Probabilidade de desempenho alto: 8.7%
Nota estimada: 470.7
